# Ejercicio 2 y 3 — Estadística descriptiva, exploración y correlaciones

Trabaja sobre la población analítica de 2025 guardada en Parquet por el ejercicio 1 (`../working_dir/parquet/eneic_2025_preparado`). Todas las métricas se calculan sobre el conjunto completo en Spark; a pandas solo se pasan tablas ya agregadas para graficar.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab7-Ej2y3")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

PARQUET_DIR = "../working_dir/parquet"
RUTA_2025 = f"{PARQUET_DIR}/eneic_2025_preparado"

df = spark.read.parquet(RUTA_2025).persist()
print(spark.version)
print("Registros en la población analítica de 2025:", df.count())
df.printSchema()

## Ejercicio 2 — Estadística descriptiva y preguntas de exploración (5 puntos)

### 2.1 Estadísticos descriptivos de salario, edad, antigüedad y horas

Cantidad de observaciones, media, mediana, desviación estándar, mínimo, máximo, percentil 25, percentil 75 y percentil 95, calculados sobre el conjunto completo con `approxQuantile` (error relativo 0) y funciones de agregación de Spark.

In [ ]:
VARIABLES = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

def resumen_descriptivo(df, columnas):
    filas = []
    for c in columnas:
        base = (df.select(
                    F.count(F.col(c)).alias("n"),
                    F.mean(c).alias("media"),
                    F.stddev(c).alias("desv_std"),
                    F.min(c).alias("minimo"),
                    F.max(c).alias("maximo"))
                .collect()[0])
        p25, mediana, p75, p95 = df.approxQuantile(c, [0.25, 0.5, 0.75, 0.95], 0.0)
        filas.append({
            "variable": c, "n": base["n"], "media": base["media"],
            "mediana": mediana, "desv_std": base["desv_std"],
            "minimo": base["minimo"], "maximo": base["maximo"],
            "p25": p25, "p75": p75, "p95": p95,
        })
    return pd.DataFrame(filas).set_index("variable").round(2)

tabla_descriptiva = resumen_descriptivo(df, VARIABLES)
tabla_descriptiva

### 2.2 Distribución por categoría ocupacional, nivel educativo y dominio

Conteos agregados en Spark; solo la tabla pequeña resultante se lleva a pandas para graficar.

In [ ]:
def conteo_categoria(df, col):
    return (df.groupBy(col).count()
              .orderBy(col)
              .toPandas())

cat_ocupacional = conteo_categoria(df, "categoria_ocupacional")
cat_nivel = conteo_categoria(df, "nivel_educativo")
cat_dominio = conteo_categoria(df, "dominio")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (data, col, titulo) in zip(axes, [
    (cat_ocupacional, "categoria_ocupacional", "Categoría ocupacional"),
    (cat_nivel, "nivel_educativo", "Nivel educativo"),
    (cat_dominio, "dominio", "Dominio"),
]):
    ax.bar(data[col].astype(str), data["count"], color="steelblue")
    ax.set_title(titulo)
    ax.set_xlabel(col)
    ax.set_ylabel("Registros")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

> **Nota:** los códigos de `categoria_ocupacional`, `nivel_educativo` y `dominio` deben leerse contra el diccionario de datos correspondiente para reemplazar el número por su etiqueta real antes de interpretar la gráfica en el informe.

### 2.3 Distribución del salario: ¿simétrica o asimétrica?

El histograma se calcula por bins en Spark (no se trae la columna completa a pandas); el boxplot usa los cuantiles ya calculados en 2.1, así que tampoco requiere los datos crudos.

In [ ]:
# Histograma en escala log10 calculado en Spark (solo cuentan bins, no filas individuales)
df_log = df.filter(F.col("salario_mensual") > 0).withColumn(
    "log10_salario", F.log10("salario_mensual"))

min_log, max_log = df_log.select(
    F.min("log10_salario"), F.max("log10_salario")).first()

n_bins = 30
ancho = (max_log - min_log) / n_bins
df_bins = df_log.withColumn(
    "bin",
    F.least(F.floor((F.col("log10_salario") - F.lit(min_log)) / F.lit(ancho)),
            F.lit(n_bins - 1))
)
hist_pd = (df_bins.groupBy("bin").count().orderBy("bin").toPandas())
hist_pd["centro_log10"] = min_log + (hist_pd["bin"] + 0.5) * ancho

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(hist_pd["centro_log10"], hist_pd["count"], width=ancho * 0.9, color="darkorange")
axes[0].set_title("Distribución del salario mensual (escala log10)")
axes[0].set_xlabel("log10(salario mensual en Q)")
axes[0].set_ylabel("Registros")

# Boxplot construido con los cuantiles ya calculados (no requiere los datos crudos)
stats_salario = {
    "label": "salario_mensual",
    "med": tabla_descriptiva.loc["salario_mensual", "mediana"],
    "q1": tabla_descriptiva.loc["salario_mensual", "p25"],
    "q3": tabla_descriptiva.loc["salario_mensual", "p75"],
    "whislo": tabla_descriptiva.loc["salario_mensual", "minimo"],
    "whishi": tabla_descriptiva.loc["salario_mensual", "p95"],
    "fliers": [],
}
axes[1].bxp([stats_salario], showfliers=False)
axes[1].set_title("Boxplot del salario mensual (Q)")
axes[1].set_ylabel("Quetzales")
plt.tight_layout()
plt.show()

print(f"Media:   Q{tabla_descriptiva.loc['salario_mensual', 'media']:,.2f}")
print(f"Mediana: Q{tabla_descriptiva.loc['salario_mensual', 'mediana']:,.2f}")

**Respuesta — simetría y diferencia media/mediana:**

_(Completa con lo que muestre tu ejecución: compara el valor de media vs. mediana de la tabla de 2.1 y la forma del histograma. Si la media es mayor que la mediana y el histograma tiene cola larga a la derecha, la distribución es asimétrica positiva — típico en variables de ingreso, donde unos pocos salarios altos jalan la media hacia arriba sin mover la mediana.)_

### 2.4 Salario mediano por nivel educativo y categoría ocupacional

In [ ]:
def mediana_por_grupo(df, col_grupo, col_valor="salario_mensual"):
    return (df.groupBy(col_grupo)
              .agg(F.percentile_approx(col_valor, 0.5).alias("salario_mediano"),
                   F.count(col_valor).alias("n"))
              .orderBy(col_grupo)
              .toPandas())

mediana_nivel = mediana_por_grupo(df, "nivel_educativo")
mediana_ocupacional = mediana_por_grupo(df, "categoria_ocupacional")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(mediana_nivel["nivel_educativo"].astype(str), mediana_nivel["salario_mediano"],
            color="seagreen")
axes[0].set_title("Salario mediano por nivel educativo")
axes[0].set_xlabel("nivel_educativo")
axes[0].set_ylabel("Salario mediano (Q)")

axes[1].bar(mediana_ocupacional["categoria_ocupacional"].astype(str),
            mediana_ocupacional["salario_mediano"], color="mediumpurple")
axes[1].set_title("Salario mediano por categoría ocupacional")
axes[1].set_xlabel("categoria_ocupacional")
axes[1].set_ylabel("Salario mediano (Q)")
plt.tight_layout()
plt.show()

mediana_nivel

### 2.5 Tamaño de muestra y salario mediano por trimestre

In [ ]:
por_trimestre = (df.groupBy("periodo_archivo")
                    .agg(F.count("*").alias("n_registros"),
                         F.percentile_approx("salario_mensual", 0.5).alias("salario_mediano"))
                    .orderBy("periodo_archivo")
                    .toPandas())

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.bar(por_trimestre["periodo_archivo"], por_trimestre["n_registros"],
        color="lightsteelblue", label="Registros")
ax1.set_ylabel("Registros")
ax1.set_xlabel("Trimestre")

ax2 = ax1.twinx()
ax2.plot(por_trimestre["periodo_archivo"], por_trimestre["salario_mediano"],
         color="crimson", marker="o", label="Salario mediano")
ax2.set_ylabel("Salario mediano (Q)")

fig.suptitle("Tamaño de la muestra analítica y salario mediano por trimestre")
fig.tight_layout()
plt.show()

por_trimestre

**Respuesta:**

_(Describe con la tabla y gráfica anteriores si el tamaño de la muestra y el salario mediano se mantienen estables entre trimestres o si hay una tendencia/caída notoria, y qué podría explicarla — por ejemplo estacionalidad del empleo o cambios en la tasa de respuesta de la encuesta.)_

## Ejercicio 3 — Relaciones entre variables numéricas (5 puntos)

Correlación de Pearson entre salario, edad, antigüedad y horas habituales, calculada con `VectorAssembler` y `Correlation.corr()` de `pyspark.ml.stat` sobre todos los registros elegibles de 2025.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

COLS_CORR = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

df_corr = df.select(COLS_CORR).na.drop()
print("Registros usados para la correlación (sin nulos en las 4 variables):", df_corr.count())

assembler = VectorAssembler(inputCols=COLS_CORR, outputCol="features")
df_vector = assembler.transform(df_corr).select("features")

matriz_corr = Correlation.corr(df_vector, "features", method="pearson").collect()[0][0]
matriz_np = matriz_corr.toArray()

matriz_pd = pd.DataFrame(matriz_np, index=COLS_CORR, columns=COLS_CORR).round(3)
matriz_pd

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(matriz_pd, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlación de Pearson entre variables numéricas (2025)")
plt.tight_layout()
plt.show()

**Respuestas:**

- ¿Qué variables presentan mayor asociación lineal con el salario? _(Mira la fila/columna `salario_mensual` de la matriz: reporta el valor más alto en valor absoluto y coméntalo — con estas variables la asociación suele ser moderada, no fuerte, porque el salario depende de muchos otros factores no incluidos aquí.)_
- ¿Existe relación entre edad y antigüedad? _(Revisa la celda `edad`–`antiguedad`: si es positiva y relativamente alta, tiene sentido porque a mayor edad hay más tiempo posible acumulado en el empleo actual, aunque la relación no es perfecta porque alguien puede cambiar de trabajo a cualquier edad.)_